In [7]:
import pandas as pd

# 1. LOAD THE DATA

In [8]:
df = pd.read_csv('data/preprocessed/cleaned_retail_data_2022_2025.csv')
rfm = pd.read_csv('data/preprocessed/rfm_base.csv', index_col='CustomerName')
item_similarity_df = pd.read_pickle('data/preprocessed/item_similarity_matrix.pkl')

b2b_rules = pd.read_csv('data/preprocessed/b2b_association_rules.csv') 
b2c_rules = pd.read_csv('data/preprocessed/b2c_association_rules.csv')

popularity = df.groupby('ProductName')['InvoiceID'].nunique().reset_index()
popularity.columns = ['ProductName', 'PurchaseFrequency']
popularity = popularity.sort_values(by='PurchaseFrequency', ascending=False)

# 2. HYBRID RECOMMENDATION

In [9]:
def get_hybrid_recommendations(customer_name, current_basket):
    """
    Combines CF, Association Rules, and Popularity based on customer type.
    """
    recommendations = {
        'Customer': customer_name,
        'Type': 'B2C (Anonymous/New)' if customer_name == 'SPRZEDAZ FISKALNA' or customer_name not in rfm.index else 'B2B (Known)',
        'Current_Basket': current_basket,
        'CF_Recs': [],
        'AR_Recs': [],
        'Popularity_Fallback': []
    }

    # --- A. COLLABORATIVE FILTERING (Only for known B2B) ---
    if recommendations['Type'] == 'B2B (Known)':
        for item in current_basket:
            if item in item_similarity_df.columns:
                sim_items = item_similarity_df[item].sort_values(ascending=False).drop(item).head(2)
                for rec_item, score in sim_items.items():
                    if score > 0: # Only add if there is some similarity
                        recommendations['CF_Recs'].append({'Item': rec_item, 'Score': round(score, 2), 'Based_On': item})
        
        # Deduplicate and sort
        recommendations['CF_Recs'] = sorted(
            [dict(t) for t in {tuple(d.items()) for d in recommendations['CF_Recs']}], 
            key=lambda x: x['Score'], reverse=True
        )[:3]

    # --- B. ASSOCIATION RULES (For both, using respective rule sets) ---
    rules_to_use = b2c_rules if recommendations['Type'] == 'B2C (Anonymous/New)' else b2b_rules
    
    if not rules_to_use.empty:
        for item in current_basket:
            # Find rules where the antecedent is the item in the basket
            matches = rules_to_use[rules_to_use['antecedents'] == item]
            for _, row in matches.head(2).iterrows():
                recommendations['AR_Recs'].append({
                    'Item': row['consequents'],
                    'Lift': round(row['lift'], 2),
                    'Confidence': round(row['confidence'], 2),
                    'Based_On': item
                })

    # --- C. POPULARITY FALLBACK (If basket is empty or no rules trigger) ---
    if not recommendations['CF_Recs'] and not recommendations['AR_Recs']:
        top_items = popularity['ProductName'].head(3).tolist()
        recommendations['Popularity_Fallback'] = top_items

    return recommendations

In [10]:
print("\n--- Testing the Hybrid Pipeline & LLM Prompt ---")

# Test Case 1: A Known Contractor buying a specific item
# (You will need to replace 'TIKKURILLA-PIGMENT CW CZARNY S(SS)' with an actual product from your CSV)
sample_contractor = 'M-ART MAREK BORTNICZUK'
sample_basket = ['TIKKURILLA-PIGMENT CW CZARNY S(SS)'] 

hybrid_results_b2b = get_hybrid_recommendations(sample_contractor, sample_basket)
print(hybrid_results_b2b)


--- Testing the Hybrid Pipeline & LLM Prompt ---
{'Customer': 'M-ART MAREK BORTNICZUK', 'Type': 'B2B (Known)', 'Current_Basket': ['TIKKURILLA-PIGMENT CW CZARNY S(SS)'], 'CF_Recs': [{'Item': 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)', 'Score': 0.87, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}, {'Item': 'TIKKURILA OPTIVA 5 BAZA A 9L', 'Score': 0.72, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}], 'AR_Recs': [{'Item': 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)', 'Lift': 49.12, 'Confidence': 0.94, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}, {'Item': 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)', 'Lift': 46.18, 'Confidence': 0.36, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}], 'Popularity_Fallback': []}


In [11]:
# ==========================================
# 3. THE LLM PROMPT GENERATOR
# ==========================================
def generate_llm_prompt(hybrid_output):
    """
    Translates the raw JSON/Dict output into a structured prompt for an LLM.
    """
    prompt = f"""
    You are an expert retail marketing assistant. Your job is to translate raw algorithmic recommendation data into a simple, actionable explanation for a store manager.
    
    Here is the data for the current customer interaction:
    - Customer Profile: {hybrid_output['Type']}
    - Customer Name/ID: {hybrid_output['Customer']}
    - Items currently in their cart: {', '.join(hybrid_output['Current_Basket'])}
    
    Algorithmic Recommendations:
    """
    
    if hybrid_output['AR_Recs']:
        prompt += "\n1. Cross-Selling (Market Basket Analysis):\n"
        for rec in hybrid_output['AR_Recs']:
            prompt += f"   - Recommend '{rec['Item']}' because they are buying '{rec['Based_On']}'. (Statistical Lift: {rec['Lift']}x more likely to buy).\n"

    if hybrid_output['CF_Recs']:
        prompt += "\n2. Contractor Patterns (Collaborative Filtering):\n"
        for rec in hybrid_output['CF_Recs']:
            prompt += f"   - Recommend '{rec['Item']}' because similar contractors who buy '{rec['Based_On']}' also frequently buy this.\n"

    if hybrid_output['Popularity_Fallback']:
        prompt += "\n3. Store Best-Sellers (Fallback):\n"
        prompt += f"   - Since we lack specific data for this interaction, recommend our overall best-sellers: {', '.join(hybrid_output['Popularity_Fallback'])}.\n"

    prompt += """
    Task: Write a 2-3 sentence summary for the floor manager. 
    Explain exactly WHAT to recommend to this customer and WHY, without using complex data science jargon like "Collaborative Filtering" or "Statistical Lift". Focus on the business logic.
    """
    
    return prompt

# ==========================================
# 4. TEST THE PIPELINE
# ==========================================
print("\n--- Testing the Hybrid Pipeline & LLM Prompt ---")

# Test Case 1: A Known Contractor buying a specific item
# (You will need to replace 'TIKKURILLA-PIGMENT CW CZARNY S(SS)' with an actual product from your CSV)
sample_contractor = 'M-ART MAREK BORTNICZUK'
sample_basket = ['TIKKURILLA-PIGMENT CW CZARNY S(SS)'] 

hybrid_results_b2b = get_hybrid_recommendations(sample_contractor, sample_basket)
llm_prompt_b2b = generate_llm_prompt(hybrid_results_b2b)

print("\n--- GENERATED LLM PROMPT (B2B Example) ---")
print(llm_prompt_b2b)

# Test Case 2: An Anonymous Walk-in Customer
hybrid_results_b2c = get_hybrid_recommendations('SPRZEDAZ FISKALNA', sample_basket)
llm_prompt_b2c = generate_llm_prompt(hybrid_results_b2c)

print("\n--- GENERATED LLM PROMPT (B2C Example) ---")
print(llm_prompt_b2c)


--- Testing the Hybrid Pipeline & LLM Prompt ---

--- GENERATED LLM PROMPT (B2B Example) ---

    You are an expert retail marketing assistant. Your job is to translate raw algorithmic recommendation data into a simple, actionable explanation for a store manager.
    
    Here is the data for the current customer interaction:
    - Customer Profile: B2B (Known)
    - Customer Name/ID: M-ART MAREK BORTNICZUK
    - Items currently in their cart: TIKKURILLA-PIGMENT CW CZARNY S(SS)
    
    Algorithmic Recommendations:
    
1. Cross-Selling (Market Basket Analysis):
   - Recommend 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)' because they are buying 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'. (Statistical Lift: 49.12x more likely to buy).
   - Recommend 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)' because they are buying 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'. (Statistical Lift: 46.18x more likely to buy).

2. Contractor Patterns (Collaborative Filtering):
   - Recommend 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)'